# D1.2 · Context that makes triage work

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.1 · From alert queue to loop operator](https://spbreed.github.io/cyber-commons/lessons/D1.1.html)**.

| | |
|---|---|
| Tools used | Wazuh, GLM-4.6, Llama 3.3, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** A/B a generic prompt vs a context-loaded one on the same alert set.

**Why a security engineer needs it.** Generic triage agents underperform your worst analyst. The control it builds is: feed the baseline, known FPs, crown-jewel map and prior decisions.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Most bad triage is not a bad model. It is an agent asked to decide without the identity, asset and history context a human analyst would have pulled without noticing they pulled it.

> **At CyberTravels.** An alert saying `cybertravels-svc listed all customer records` is untriageable without knowing whether that is its job. Most bad triage at CyberTravels is missing context, not a weak model.

## 2 · The framework

```
   the alert                what a human would have pulled without thinking
   +----------------+       +-----------------------------------+
   | user: dana     |  -->  | is dana on call?                  |
   | host: build-07 |       | is build-07 a build agent?        |
   | 03:14          |       | has this fired for dana before?   |
   +----------------+       +-----------------------------------+

   most bad triage is missing context, not a weak model
```

An alert about a human is triageable with three facts: who, what, when. An alert
about an agent needs three more, and without them every analyst has to guess.

- **The acting identity** and the principal it acted for (A2.1).
- **The scopes it held** at the time. This is the decisive field: reading
  `.env` is alarming for an agent scoped `repo:read` and routine for a
  secrets-rotation agent.
- **The delegation chain**, so the analyst can see who caused the task.

Without scope in the alert, the analyst's only options are to escalate
everything or to develop a habit of closing agent alerts. Both happen, and the
second one happens quietly.

## 3 · Triage as a skill — and the sample that keeps it honest

Context turns a guess into a verdict. Automating the verdict without automating the audit of it is how a closing rule quietly starts closing real incidents.

The skill therefore requires a sampling rule over anything auto-closed, and requires its seed to come from something **stable**. Sampling seeded from `hash()` picks a different subset on every run, so you can never tell whether a change in findings came from the rule or from the dice.

In [ ]:
# skills/secops/detection-triage/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: detection-triage
description: >-
  Triage security alerts with the context needed to reach a defensible verdict,
  and sample what is auto-closed so the closing rule stays honest. Use when
  working an alert queue, deciding whether an alert is a true positive, tuning
  a noisy detection, or designing automated alert handling.
allowed-tools: Read, Grep, Bash
---

# Alert triage with context

An analyst reading an alert in isolation is guessing. The verdict comes from
the alert **plus** the context that makes it normal or abnormal, and most
triage automation fails because it automated the guess instead of the context.

## When to use this

Working an alert queue, building an auto-close rule, or reviewing why a
detection produces verdicts nobody trusts.

## Procedure

**1 — Gather the context before judging.** For every alert, assemble:

- **asset** — what it is, who owns it, how exposed it is
- **identity** — human or workload, and its normal behaviour
- **history** — has this fired before on this asset, and how was it resolved
- **change** — was there a deploy, a migration, a new agent, in the window
- **peers** — did the same thing fire elsewhere at the same time

A verdict issued without `history` will re-litigate a decision the team already
made, which is the most common way triage automation loses trust.

**2 — Reach a verdict, with the reason.** One of `true_positive`,
`false_positive`, `benign_true_positive` (it really happened and it is fine),
or `needs_human`. Record which context field decided it. "Benign true positive"
is a distinct category and collapsing it into false-positive corrupts every
tuning decision made from the data afterwards.

**3 — Attach confidence, and let it gate automation.** Only high-confidence
verdicts may auto-close. Everything else queues.

**4 — Sample the auto-closed.** Automation that closes alerts must be audited
by re-opening a fraction of them for human review. This is the control that
catches a closing rule that has quietly started closing real incidents.

Seed the sampler from something **stable** — a checksum of the alert id, never
`hash()`, which Python randomises per process. A sampling rule that picks a
different subset every run cannot be audited, because you cannot tell whether a
change in findings came from the rule or from the dice.

**5 — Feed tuning from verdicts, not volume.** A detection is noisy if its
false-positive rate is high, not if it fires often. Rank tuning candidates by
`false_positive_rate × volume`, with a full tiebreak so the list is stable
between runs.

## Output contract

```json
{
  "triaged": [
    {"alert_id": "str", "verdict": "true_positive|false_positive|benign_true_positive|needs_human",
     "deciding_context": "asset|identity|history|change|peers",
     "reason": "str", "confidence": 0.0,
     "auto_closed": false, "sampled_for_review": false}
  ],
  "sampling": {"rate": 0.0, "seed_source": "str", "reviewed": 0, "disagreements": 0},
  "tuning": [{"rule": "str", "fp_rate": 0.0, "volume": 0, "priority": 0.0}]
}
```

`disagreements` is the number that matters: it is the measured error rate of
the automation, and it belongs in every report about it.

## Failure modes

- **Auto-closing without sampling.** The rule then has no error bar and no way
  to acquire one.
- **Seeding the sampler from `hash()`.** Non-reproducible sampling is not a
  control.
- **Folding benign-true-positive into false-positive.** It teaches the tuner to
  suppress a working detection.
- **Removing `needs_human`.** Forced verdicts under uncertainty are how a queue
  becomes an incident.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/secops/detection-triage/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/secops/detection-triage/scripts/detection_triage.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Assemble the context a triage decision needs, and show what the same alert looks like without it.

This is the executable half of the `detection-triage` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

# --- the skill's own contract, available both ways -------------------------
# This script is run two ways and both have to work: standalone from a
# terminal, and embedded in the lesson notebook underneath the cell that
# already parsed the SKILL.md. So take what is already defined and read the
# file only when it is not.
import pathlib as _pathlib


def _skill_md():
    if "SKILL_MD" in globals():
        return globals()["SKILL_MD"]
    return (_pathlib.Path(__file__).resolve().parent.parent / "SKILL.md").read_text()


if "contract_of" not in globals():
    import json, re

    def parse_skill(md):
        """Split a SKILL.md into (frontmatter dict, body).

        Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
        scalars (`description: >-`) whose continuation lines are indented. That is
        all a skill needs, and parsing it directly means no dependency.
        """
        if not md.startswith("---"):
            raise ValueError("a SKILL.md must open with a frontmatter block")
        _, front, body = md.split("---", 2)
        meta, key = {}, None
        for line in front.strip().splitlines():
            if not line.strip():
                continue
            if not line[0].isspace() and ":" in line:
                key, val = line.split(":", 1)
                key, val = key.strip(), val.strip()
                # `>-` and `|` open a folded block; the value is on the next lines
                meta[key] = "" if val in (">-", ">", "|", "|-") else val
            elif key is not None:
                meta[key] = (meta[key] + " " + line.strip()).strip()
        if "allowed-tools" in meta:
            meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                     if t.strip()]
        for required in ("name", "description"):
            if not meta.get(required):
                raise ValueError(f"skill is missing a {required!r}")
        return meta, body.strip()

    _WORD = re.compile(r"[a-z][a-z-]{3,}")

    def route(task, skills):
        """Pick the skill whose description best matches a task. Deterministic.

        The description is not documentation — it is the routing key. An agent
        decides whether to load a skill by reading it, so a vague description means
        the skill never fires when it should, and two overlapping descriptions mean
        the wrong one fires.

        Returns (pick, scores, margin). A margin of 0 means the top two scored the
        same and the "winner" is just whichever sorted first — an arbitrary answer
        wearing a confident face. Callers should refuse to auto-route on margin 0
        rather than pretend the tiebreak meant something.
        """
        want = set(_WORD.findall(task.lower()))
        def score(meta):
            return len(want & set(_WORD.findall(meta["description"].lower())))
        scores = {n: score(skills[n]) for n in sorted(skills)}
        # sort names first, then by score: ties must break identically on every
        # machine or the same task routes differently on two runs
        ranked = sorted(sorted(skills), key=lambda n: -scores[n])
        top = scores[ranked[0]]
        margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
        return ranked[0], scores, margin

    def contract_of(body):
        """The JSON block under '## Output contract' — the skill's machine promise."""
        # non-greedy across any prose between the heading and the fence
        m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
        if not m:
            raise ValueError("skill declares no output contract")
        return json.loads(m.group(1))

    def check(instance, contract, path="$"):
        """Structural conformance of an instance against a contract template.

        Returns the list of problems. An empty list means the shape is right — and
        that is *all* it means. Conformance is not accuracy: an empty findings list
        conforms perfectly and tells you nothing.
        """
        problems = []
        if isinstance(contract, dict):
            if not isinstance(instance, dict):
                return [f"{path}: expected an object, got {type(instance).__name__}"]
            for k, v in sorted(contract.items()):
                if k not in instance:
                    problems.append(f"{path}.{k}: missing")
                else:
                    problems += check(instance[k], v, f"{path}.{k}")
        elif isinstance(contract, list):
            if not isinstance(instance, list):
                return [f"{path}: expected a list, got {type(instance).__name__}"]
            for i, item in enumerate(instance):          # every element, same template
                problems += check(item, contract[0], f"{path}[{i}]")
        elif isinstance(contract, str) and "|" in contract:
            if instance not in contract.split("|"):
                problems.append(f"{path}: {instance!r} is not one of {contract}")
        elif isinstance(contract, bool):                  # before the numeric case:
            if not isinstance(instance, bool):            # bool is a subclass of int
                problems.append(f"{path}: expected bool, got {type(instance).__name__}")
        elif isinstance(contract, (int, float)):
            # JSON has one number type. A contract written `0` must accept 0.4, or
            # every cost and rate in the pipeline has to be rounded to satisfy a
            # checker rather than to be correct.
            if isinstance(instance, bool) or not isinstance(instance, (int, float)):
                problems.append(f"{path}: expected a number, got {type(instance).__name__}")
        elif not isinstance(instance, type(contract)):
            problems.append(f"{path}: expected {type(contract).__name__}, "
                            f"got {type(instance).__name__}")
        return problems

SKILL_MD = _skill_md()
meta, body = parse_skill(SKILL_MD)

import time
from dataclasses import dataclass, field

@dataclass
class Token:
    sub: str; actor: str; scopes: set; act: dict = None
    def chain(self):
        out, node = [], self.act
        while node: out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub: c.insert(0, self.sub)
        return c

AGENTS = {
 "patch-agent":   Token("dana@corp", "patch-agent", {"repo:read", "repo:write"},
                        {"actor": "orchestrator", "act": None}),
 "rotator-agent": Token("ops@corp", "rotator-agent", {"secrets:read", "secrets:write"},
                        {"actor": "scheduler", "act": None}),
}
EVENT = {"action": "read_file", "target": "/vault/.env", "ts": time.time()}

print("BARE ALERT (what most SOCs receive):")
for actor in AGENTS:
    print(f"   {actor} read {EVENT['target']}")
print("   → identical. An analyst cannot tell these apart.\n")

print("ENRICHED ALERT:")
for actor, tok in AGENTS.items():
    expected = "secrets:read" in tok.scopes
    print(f"   actor        {tok.actor}")
    print(f"   on behalf of {tok.sub}")
    print(f"   chain        {' → '.join(tok.chain())}")
    print(f"   scopes held  {sorted(tok.scopes)}")
    print(f"   verdict      {'EXPECTED — this agent rotates secrets' if expected else 'ANOMALY — no secrets scope'}")
    print()

def triage_without_context(event):
    """All the analyst has is the action and the target."""
    return "escalate" if "/.env" in event["target"] or "secret" in event["target"] else "close"

def triage_with_context(event, token):
    needed = "secrets:read"
    if "/.env" in event["target"] or "vault" in event["target"]:
        return "close" if needed in token.scopes else "escalate"
    return "close"

TRUTH = {"patch-agent": "tp", "rotator-agent": "fp"}
print(f"{'agent':16s}{'no context':14s}{'with context':16s}{'truth':>7}")
print("-" * 56)
for actor, tok in AGENTS.items():
    a = triage_without_context(EVENT)
    b = triage_with_context(EVENT, tok)
    print(f"{actor:16s}{a:14s}{b:16s}{TRUTH[actor]:>7}")

print("\nWithout scopes, both escalate → the rotator generates a false positive")
print("every single night, and within a month the rule is tuned off.")

FIELDS = {
 "acting identity":  "which agent — not the human whose token it borrowed",
 "principal":        "who the action was for",
 "delegation chain": "who caused the task; where to look for the trigger",
 "scopes held":      "THE decisive field — is this action within its remit?",
 "tool + target":    "what it did",
 "session/trace id": "so the analyst can pull the whole run (D1.5)",
}
for k, v in FIELDS.items(): print(f"{k:20s}{v}")

def enrich(event, token, trace_id):
    return {"acting_identity": token.actor, "principal": token.sub,
            "chain": " → ".join(token.chain()), "scopes": sorted(token.scopes),
            "tool": event["action"], "target": event["target"],
            "trace_id": trace_id,
            "within_remit": any(s.startswith("secrets") for s in token.scopes)
                            if "vault" in event["target"] or "/.env" in event["target"]
                            else True}

print()
for actor, tok in AGENTS.items():
    e = enrich(EVENT, tok, trace_id=f"tr-{actor[:4]}-8812")
    verdict = "close (within remit)" if e["within_remit"] else "ESCALATE (outside remit)"
    print(f"{actor:16s}{verdict}")
    print(f"{'':16s}{e['chain']}  scopes={e['scopes']}")

assert enrich(EVENT, AGENTS["patch-agent"], "x")["within_remit"] is False
assert enrich(EVENT, AGENTS["rotator-agent"], "x")["within_remit"] is True

contract = contract_of(body)
import zlib

ALERTS = [("A-1", "patch-agent"), ("A-2", "rotator-agent"), ("A-3", "patch-agent"),
          ("A-4", "rotator-agent"), ("A-5", "patch-agent"), ("A-6", "rotator-agent")]
SAMPLE_RATE = 0.34

def sampled(alert_id):
    # crc32, never hash(): Python randomises string hashing per process, so a
    # hash()-seeded sampler is unauditable by construction
    return (zlib.crc32(alert_id.encode()) % 100) < SAMPLE_RATE * 100

triaged = []
for aid, agent in ALERTS:
    ctx = enrich(EVENT, AGENTS[agent], aid)
    within = ctx["within_remit"]
    triaged.append({
      "alert_id": aid,
      # reading /vault/.env is real either way; whether it is *fine* is decided
      # by the acting identity's remit, which is the context field that matters
      "verdict": "benign_true_positive" if within else "true_positive",
      "deciding_context": "identity",
      "reason": f"{ctx['acting_identity']} scopes {ctx['scopes']}",
      "confidence": 0.9 if within else 0.95,
      "auto_closed": within,
      "sampled_for_review": within and sampled(aid)})

reviewed = [t for t in triaged if t["sampled_for_review"]]
triage = {
 "triaged": triaged,
 "sampling": {"rate": SAMPLE_RATE, "seed_source": "zlib.crc32(alert_id)",
              "reviewed": len(reviewed), "disagreements": 0},
 "tuning": [{"rule": "vault-file-read", "fp_rate": 0.0,
             "volume": len(ALERTS), "priority": 0.0}],
}
problems = check(triage, contract)
print(f"conformance: {len(problems)} problem(s)")
for p in problems: print("   ", p)
assert not problems, problems

for t in triaged:
    print(f"   {t['alert_id']}  {t['verdict']:20s} auto_closed={str(t['auto_closed']):5s} "
          f"sampled={t['sampled_for_review']}")
print(f"\nauto-closed {sum(t['auto_closed'] for t in triaged)}, "
      f"of which {len(reviewed)} sampled back for human review")
print()
print("`benign_true_positive` is its own verdict. Folding it into false-positive")
print("would teach the tuner to suppress a detection that is working exactly as")
print("designed - the read really happened, it was simply within remit.")
assert any(t["verdict"] == "benign_true_positive" for t in triaged)
assert any(t["verdict"] == "true_positive" for t in triaged)
assert reviewed, "auto-closing with no sample is a rule with no error bar"

## What you just proved

The bare alert is identical for both agents. Enriched, the secrets-rotation agent is within remit and the patch agent is not. Context-free triage escalates both — generating a nightly false positive — while scope-aware triage matches ground truth on both.

## Your turn

Check which of the six fields your agent telemetry carries today. Scopes-held is the one almost nobody logs, and it is the one that decides the alert.

---

**Next → [D1.3 · Agent-assisted detection engineering](https://spbreed.github.io/cyber-commons/lessons/D1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*